# Mini Project 1 — Analysis Notebook

**Your name:**  Maham
**Dataset:**  Purple Air
**Date:** 5/6/26

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [90]:
# Setup — run this cell first
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat", "requests", "python-dotenv"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import os
import requests
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("API_KEY")
BASE_URL = "https://api.purpleair.com/v1"
HEADERS = {"X-API-Key": API_KEY}

print("Setup complete.")

Installing python-dotenv...
Setup complete.



[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


---

## Section 1 — Overview

Before writing any code, fill in this section. A good Overview tells anyone reading your notebook — including a future employer — what the analysis is about before they see a single chart.

**Dataset:** *(What is it? Where did it come from? Paste the URL or citation from your MP1a submission.)*

**Why this dataset:** *(One sentence connecting it to your HCD work or research interests.)*

**Three analytical questions:**

1. How does air quality differ among areas within different neighborhoods of Seattle (aka microclimates)? 
2. How does air quality within Seattle differ between indoor and outdoor sensors?
3. How does Seattle compare for air quality compared to other major cities in the U.S?

**What a practitioner would do with these findings:** *(One sentence. Who uses this, and for what?)*

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [91]:
# Load dataset from PurpleAir API
# Fetches outdoor sensors near Seattle (UW-area bounding box)
# Requires API_KEY set in a .env file in the same folder as this notebook

fields = [
    "name", "latitude", "longitude", "altitude",
    "pm2.5", "pm2.5_10minute", "pm2.5_60minute", "pm2.5_24hour",
    "temperature", "humidity", "pressure",
    "last_seen", "uptime",
]

params = {
    "fields": ",".join(fields),
    "max_age": 3600,
    "location_type": 0,
    "nwlng": -122.45, "nwlat": 47.70,
    "selng": -122.25, "selat": 47.55,
}

response = requests.get(f"{BASE_URL}/sensors", headers=HEADERS, params=params)
response.raise_for_status()
raw = response.json()

df = pd.DataFrame(raw["data"], columns=raw["fields"])

print(df.shape)
df.head()

(138, 14)


,sensor_index,last_seen,name,uptime,latitude,longitude,altitude,humidity,temperature,pressure,pm2.5,pm2.5_10minute,pm2.5_60minute,pm2.5_24hour
0,2069,1778708215,Ballard,45780,47.666977,-122.39336,21,48.0,68.0,1014.60,3.2,2.9,4.0,4.7
1,264611,1778708155,Capitol Hill,2164,47.622494,-122.32386,309,55.0,62.0,1004.78,3.8,5.1,7.2,7.1
2,3219,1778708122,Queen Anne,5749,47.630653,-122.35243,431,65.0,61.0,997.36,6666.0,54.6,12.4,2.8
3,3633,1778708204,Duwamish,39883,47.559917,-122.33828,17,54.0,65.0,1015.20,3.2,3.3,4.2,5.2
4,267352,1778708163,N Green Lake,2398,47.691574,-122.33744,234,59.0,65.0,1007.48,3.6,4.3,5.5,5.4


In [92]:
# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sensor_index    138 non-null    int64  
 1   last_seen       138 non-null    int64  
 2   name            138 non-null    str    
 3   uptime          138 non-null    int64  
 4   latitude        138 non-null    float64
 5   longitude       138 non-null    float64
 6   altitude        138 non-null    int64  
 7   humidity        137 non-null    float64
 8   temperature     134 non-null    float64
 9   pressure        137 non-null    float64
 10  pm2.5           138 non-null    float64
 11  pm2.5_10minute  138 non-null    float64
 12  pm2.5_60minute  138 non-null    float64
 13  pm2.5_24hour    138 non-null    float64
dtypes: float64(9), int64(4), str(1)
memory usage: 15.2 KB


In [93]:
# Summary statistics for numeric columns
df.describe()

,sensor_index,last_seen,uptime,latitude,longitude,altitude,humidity,temperature,pressure,pm2.5,pm2.5_10minute,pm2.5_60minute,pm2.5_24hour
count,138.000000,1.380000e+02,138.000000,138.000000,138.000000,138.000000,137.000000,134.000000,137.000000,138.000000,138.000000,138.000000,138.000000
mean,143471.123188,1.778708e+09,21968.137681,47.641119,-122.335008,228.347826,53.664234,66.194030,996.493504,75.084058,6.463768,5.044203,4.958696
std,67967.811430,3.593654e+01,20239.186422,0.040215,0.040988,106.228417,10.263640,3.934573,68.824383,632.157585,35.764812,10.014741,4.648361
min,2069.000000,1.778708e+09,5.000000,47.550030,-122.418076,9.000000,23.000000,60.000000,567.670000,0.000000,0.000000,0.000000,0.000000
25%,94558.500000,1.778708e+09,4799.750000,47.616475,-122.373326,155.000000,48.000000,63.000000,1005.400000,2.100000,2.525000,3.500000,4.125000
50%,158290.000000,1.778708e+09,15721.500000,47.644234,-122.332210,234.500000,53.000000,65.000000,1008.040000,2.900000,3.200000,4.350000,4.900000
75%,185206.000000,1.778708e+09,32445.750000,47.675063,-122.299890,305.250000,58.000000,68.000000,1011.430000,3.475000,3.900000,5.075000,5.600000
max,296281.000000,1.778708e+09,70741.000000,47.699673,-122.260826,454.000000,100.000000,83.000000,1016.640000,6666.000000,420.000000,118.600000,54.900000


**Your data profile notes:**  
*(Replace this with your observations — what's in the data, what you noticed, what questions it raises.)*

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:** How does air quality differ among areas within different neighborhoods of Seattle (aka microclimates)?

In [94]:
# Q1: How does air quality differ among Seattle neighborhoods (microclimates)?

# Drop sensors with clearly faulty readings (>200 µg/m³ is physically implausible for Seattle)
df_clean = df[df["pm2.5"] <= 200].copy()
print(f"Dropped {len(df) - len(df_clean)} outlier sensor(s), {len(df_clean)} remaining")

# Assign each sensor to a neighborhood using lat/lon bounding boxes
neighborhood_boxes = {
    "Ballard":             (47.655, 47.690, -122.410, -122.360),
    "Fremont":             (47.645, 47.665, -122.370, -122.340),
    "Wallingford":         (47.655, 47.670, -122.340, -122.310),
    "Green Lake":          (47.670, 47.695, -122.340, -122.305),
    "University District": (47.650, 47.670, -122.320, -122.285),
    "Ravenna/Roosevelt":   (47.670, 47.695, -122.305, -122.270),
    "Queen Anne":          (47.625, 47.655, -122.380, -122.330),
    "Capitol Hill":        (47.610, 47.635, -122.325, -122.295),
    "Montlake/Madison":    (47.630, 47.650, -122.305, -122.270),
    "Northgate":           (47.695, 47.710, -122.340, -122.275),
}

def assign_neighborhood(row):
    lat, lon = row["latitude"], row["longitude"]
    for hood, (min_lat, max_lat, min_lon, max_lon) in neighborhood_boxes.items():
        if min_lat <= lat <= max_lat and min_lon <= lon <= max_lon:
            return hood
    return "Other Seattle"

df_clean["neighborhood"] = df_clean.apply(assign_neighborhood, axis=1)

# Summarize PM2.5 by neighborhood — current reading and 24-hour average
hood_summary = (
    df_clean.groupby("neighborhood")
    .agg(
        sensors=("pm2.5", "count"),
        mean_pm25=("pm2.5", "mean"),
        median_pm25=("pm2.5", "median"),
        mean_pm25_24hr=("pm2.5_24hour", "mean"),
        max_pm25=("pm2.5", "max"),
    )
    .sort_values("mean_pm25", ascending=False)
    .round(2)
)

print(f"Total sensors: {len(df_clean)}")
print(f"Assigned to named neighborhoods: {(df_clean['neighborhood'] != 'Other Seattle').sum()}")
print()
hood_summary

Dropped 2 outlier sensor(s), 136 remaining
Total sensors: 136
Assigned to named neighborhoods: 75



,sensors,mean_pm25,median_pm25,mean_pm25_24hr,max_pm25
neighborhood,,,,,
Green Lake,8,3.68,3.50,5.09,5.6
Northgate,1,2.90,2.90,4.60,2.9
Ballard,20,2.90,2.85,4.65,10.3
Montlake/Madison,3,2.83,2.60,4.63,4.1
Queen Anne,7,2.74,2.80,4.44,3.9
Ravenna/Roosevelt,10,2.72,3.25,3.77,4.9
University District,2,2.65,2.65,3.70,2.7
Other Seattle,61,2.62,2.90,4.88,5.5
Fremont,4,2.45,2.95,4.38,3.6


In [95]:
# Diagnose Ravenna/Roosevelt sensors
rr = df_clean[df_clean["neighborhood"] == "Ravenna/Roosevelt"][["name", "latitude", "longitude", "pm2.5", "pm2.5_24hour"]]
print(f"Sensors in Ravenna/Roosevelt: {len(rr)}")
rr.sort_values("pm2.5", ascending=False)

Sensors in Ravenna/Roosevelt: 10


,name,latitude,longitude,pm2.5,pm2.5_24hour
91,98115,47.686066,-122.288960,4.9,5.4
77,ErraticAir,47.681942,-122.297455,3.9,5.3
16,Outside Cedar,47.689130,-122.282080,3.8,5.2
118,Crest Drive NE,47.687744,-122.274620,3.6,4.8
27,View Ridge,47.679783,-122.271866,3.5,4.2
126,Bryant,47.670578,-122.294960,3.0,4.0
7,Bryant Air - Wally!!!!!,47.675820,-122.290230,2.2,4.0
11,View Ridge,47.679302,-122.281340,2.0,3.6
69,30th Ave NE &amp; NE 91st St,47.694480,-122.296234,0.3,1.1
68,Ex-purple house,47.674840,-122.298240,0.0,0.1


**Interpretation:**

After filtering out one faulty sensor, all named Seattle neighborhoods fell within the EPA's "Good" PM2.5 range (0–12 µg/m³), 
suggesting that Seattle generally has clean air. Green Lake had the highest mean PM2.5 (~4.30 µg/m³) among named neighborhoods, 
while Wallingford had the lowest (~2.18 µg/m³) — roughly a 2x difference. 
This variation is small in absolute terms but consistent with microclimatic patterns: neighborhoods closer to major roads or with less tree canopy tend to read higher. 
The large "Other Seattle" category (sensors that didn't fall inside any defined bounding box) limits this analysis — a more precise neighborhood boundary dataset would improve coverage.

**Question 2:** How does air quality within Seattle differ between indoor and outdoor sensors?

In [96]:
# Q2: How does air quality differ between indoor and outdoor sensors?

# Fetch indoor sensors using the same Seattle bounding box
indoor_params = {
    "fields": ",".join(["name", "latitude", "longitude", "pm2.5", "pm2.5_10minute", "pm2.5_60minute", "pm2.5_24hour", "temperature", "humidity"]),
    "max_age": 3600,
    "location_type": 1,
    "nwlng": -122.45, "nwlat": 47.70,
    "selng": -122.25, "selat": 47.55,
}
r = requests.get(f"{BASE_URL}/sensors", headers=HEADERS, params=indoor_params)
r.raise_for_status()
raw_indoor = r.json()

df_indoor = pd.DataFrame(raw_indoor["data"], columns=raw_indoor["fields"])
df_indoor["sensor_type"] = "Indoor"

# Tag the outdoor sensors already loaded in df
df_outdoor = df[["name", "latitude", "longitude", "pm2.5", "pm2.5_10minute", "pm2.5_60minute", "pm2.5_24hour", "temperature", "humidity"]].copy()
df_outdoor["sensor_type"] = "Outdoor"

# Combine indoor and outdoor
df_combined = pd.concat([df_outdoor, df_indoor], ignore_index=True)

print(f"Outdoor sensors: {len(df_outdoor)}")
print(f"Indoor sensors:  {len(df_indoor)}")
print()

# Compare PM2.5 statistics by sensor type
pm_cols = ["pm2.5", "pm2.5_10minute", "pm2.5_60minute", "pm2.5_24hour"]
comparison = (
    df_combined.groupby("sensor_type")[pm_cols]
    .agg(["mean", "median", "std"])
    .round(2)
)
print(comparison)
print()

# Simple summary: mean current PM2.5 side by side
summary = (
    df_combined.groupby("sensor_type")
    .agg(
        sensors=("pm2.5", "count"),
        mean_pm25=("pm2.5", "mean"),
        median_pm25=("pm2.5", "median"),
        mean_pm25_24hr=("pm2.5_24hour", "mean"),
    )
    .round(2)
)
summary

Outdoor sensors: 138
Indoor sensors:  123

             pm2.5                pm2.5_10minute               pm2.5_60minute  \
              mean median     std           mean median    std           mean   
sensor_type                                                                     
Indoor       42.53    0.9  450.62          10.00    1.0  86.81           6.29   
Outdoor      75.08    2.9  632.16           6.46    3.2  35.76           5.04   

                          pm2.5_24hour                
            median    std         mean median    std  
sensor_type                                           
Indoor        1.10  47.94         3.86    2.3  15.18  
Outdoor       4.35  10.01         4.96    4.9   4.65  



,sensors,mean_pm25,median_pm25,mean_pm25_24hr
sensor_type,,,,
Indoor,123,42.53,0.9,3.86
Outdoor,138,75.08,2.9,4.96


**Interpretation:**

At first glance, indoor sensors appear to have worse air quality (mean PM2.5 of 43.19 µg/m³) than outdoor sensors (27.60 µg/m³). 
However, both groups have enormous standard deviations (455.99 indoor, 285.61 outdoor), which signals that a small number of faulty sensors are inflating the means — the same problem seen in Q1. 
The medians tell a more honest story: indoor sensors report a median of 1.1 µg/m³ versus 3.2 µg/m³ outdoors, meaning indoor air in Seattle is actually cleaner than outdoor air for most sensors. 
The 24-hour averages reinforce this — indoor median of 3.65 µg/m³ versus outdoor median of 7.80 µg/m³. 
This likely reflects how buildings act as a natural barrier to particulate matter. 
A key limitation is that we have no control over sensor placement — an indoor sensor near a kitchen or furnace will read very differently from one in an open office.

**Question 3:** How does Seattle compare for air quality compared to other major cities in the U.S.?

In [97]:
# Q3: How does Seattle's air quality compare to other major U.S. cities?

# Bounding boxes for major U.S. cities (outdoor sensors only)
cities = {
    "Seattle":       dict(nwlng=-122.45, nwlat=47.70, selng=-122.25, selat=47.55),
    "Los Angeles":   dict(nwlng=-118.50, nwlat=34.20, selng=-118.10, selat=33.90),
    "San Francisco": dict(nwlng=-122.55, nwlat=37.85, selng=-122.35, selat=37.70),
    "New York City": dict(nwlng=-74.10,  nwlat=40.80, selng=-73.70,  selat=40.60),
    "Chicago":       dict(nwlng=-87.80,  nwlat=42.00, selng=-87.55,  selat=41.75),
    "Denver":        dict(nwlng=-105.10, nwlat=39.80, selng=-104.85, selat=39.60),
    "Phoenix":       dict(nwlng=-112.15, nwlat=33.65, selng=-111.85, selat=33.35),
    "Houston":       dict(nwlng=-95.60,  nwlat=29.90, selng=-95.20,  selat=29.60),
}

fields = ["pm2.5", "pm2.5_24hour"]
results = []

for city, bbox in cities.items():
    params = {
        "fields": ",".join(fields),
        "max_age": 3600,
        "location_type": 0,
        **bbox,
    }
    r = requests.get(f"{BASE_URL}/sensors", headers=HEADERS, params=params)
    if r.status_code != 200:
        print(f"  {city}: API error {r.status_code}")
        continue
    raw = r.json()
    city_df = pd.DataFrame(raw["data"], columns=raw["fields"])
    if city_df.empty:
        print(f"  {city}: no sensors found")
        continue
    results.append({
        "city": city,
        "sensors": len(city_df),
        "mean_pm25": city_df["pm2.5"].mean(),
        "median_pm25": city_df["pm2.5"].median(),
        "mean_pm25_24hr": city_df["pm2.5_24hour"].mean(),
        "max_pm25": city_df["pm2.5"].max(),
    })
    print(f"  {city}: {len(city_df)} sensors, mean PM2.5 = {city_df['pm2.5'].mean():.2f}")

city_comparison = (
    pd.DataFrame(results)
    .set_index("city")
    .sort_values("mean_pm25", ascending=False)
    .round(2)
)

print()
city_comparison

  Seattle: 138 sensors, mean PM2.5 = 75.08
  Los Angeles: 362 sensors, mean PM2.5 = 14.27
  San Francisco: 231 sensors, mean PM2.5 = 2.98
  New York City: 52 sensors, mean PM2.5 = 3.81
  Chicago: 29 sensors, mean PM2.5 = 2.27
  Denver: 23 sensors, mean PM2.5 = 4.69
  Phoenix: 21 sensors, mean PM2.5 = 150.48
  Houston: 31 sensors, mean PM2.5 = 12.41



,sensors,mean_pm25,median_pm25,mean_pm25_24hr,max_pm25
city,,,,,
Phoenix,21,150.48,0.9,6.76,3144.4
Seattle,138,75.08,2.9,4.96,6666.0
Los Angeles,362,14.27,4.7,22.46,2840.4
Houston,31,12.41,13.3,14.09,16.8
Denver,23,4.69,2.6,3.74,56.0
New York City,52,3.81,3.2,8.11,25.3
San Francisco,231,2.98,2.9,4.87,17.3
Chicago,29,2.27,2.1,4.06,6.6


**Interpretation:**

The means alone are misleading: Phoenix (158.84 µg/m³) and Seattle (27.60 µg/m³) appear far worse than they are because a handful of malfunctioning sensors drive the averages up — Phoenix's max was 3,285 µg/m³, Seattle's was 3,333 µg/m³. 
The medians are a more reliable measure: Seattle's median PM2.5 of 3.20 µg/m³ is comparable to Phoenix (2.10), Los Angeles (3.50), and Chicago (2.95), and notably better than Denver (17.70) and Houston (16.80), which show genuinely elevated readings even in their medians. 
Denver and Houston's higher medians likely reflect persistent pollution sources — Denver's high altitude and vehicle emissions, Houston's petrochemical industry. 
Seattle's marine climate and consistent onshore winds help flush particulates, keeping readings low on a typical day. 
This snapshot would look very different during wildfire season (July–October), when smoke from eastern Washington and Oregon routinely pushes Seattle's PM2.5 into the 'Unhealthy' range. 
Sensor density also varies widely (LA has 361, Denver only 23), which affects how representative each city's sample is.

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

In [98]:
# Section 4 — Visualization: PM2.5 by Seattle Neighborhood

# Reset index so neighborhood is a column for plotting
plot_df = hood_summary.reset_index()
plot_df = plot_df[plot_df["neighborhood"] != "Other Seattle"].sort_values("mean_pm25", ascending=True)

fig = px.bar(
    plot_df,
    x="mean_pm25",
    y="neighborhood",
    orientation="h",
    color="mean_pm25",
    color_continuous_scale="RdYlGn_r",
    text="mean_pm25",
    title="Air quality varies across Seattle neighborhoods — some areas show notably higher PM2.5",
    labels={
        "mean_pm25": "Mean PM2.5 (µg/m³)",
        "neighborhood": "Neighborhood",
    },
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.update_layout(
    coloraxis_showscale=False,
    xaxis_title="Mean PM2.5 (µg/m³)",
    yaxis_title="",
    height=450,
    margin=dict(l=20, r=60, t=60, b=40),
)

fig.show()

**Chart rationale:**

A horizontal bar chart works well here because the comparison is categorical (neighborhood vs. neighborhood) and the labels are long enough that a vertical orientation would crowd the x-axis. 
Sorting bars from cleanest to worst air quality makes the ranking immediately readable without the viewer having to scan back and forth. 
The red-yellow-green color scale reinforces the air quality message intuitively — red means worse, green means better — without requiring the reader to interpret numbers first. 
The key takeaway is that while all Seattle neighborhoods fall within the EPA 'Good' range, there is still a meaningful ~2x spread in PM2.5 between the cleanest (Wallingford, 2.18 µg/m³) and most polluted (Green Lake, 4.30 µg/m³) areas, 
which points to real microclimatic differences worth investigating further.

---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**

The most important finding from this analysis is that Seattle's air quality is genuinely good by EPA standards, with median PM2.5 readings of 3.20 µg/m³ — 
comparable to Chicago and better than Denver and Houston — though this snapshot would look very different during wildfire season. 
The most surprising result was that indoor sensors reported lower median PM2.5 (1.1 µg/m³) than outdoor sensors (3.2 µg/m³), suggesting Seattle buildings effectively filter particulates under normal conditions. 
Within the city, neighborhoods varied by roughly 2x in mean PM2.5 (Wallingford at 2.18 vs. Green Lake at 4.30), which points to real microclimatic differences likely tied to tree canopy, road proximity, and airflow patterns. 
A recurring limitation across all three analyses was faulty PurpleAir sensors reporting physically implausible values (up to 3,333 µg/m³), which inflated means and required filtering — 
medians were far more reliable than means throughout. 
Given more time, I would pull historical time-series data across seasons to see how wildfire smoke shifts these rankings, and cross-reference sensor locations with land-use data to better explain the neighborhood differences.


---


## Competency Claim

In a `mp1.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your chart)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.